In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col

base_path = "abfss://racingcar@selfkazimdatabrick.dfs.core.windows.net/dataset_csv/"

# F1 Dataset ki saari 14 files ki list
f1_tables = [
    "circuits",
    "constructors",
    "drivers",
    "races",
    "results",
    "status",
    "seasons",
    "qualifying",
    "driver_standings",
    "constructor_standings",
    "constructor_results",
    "pit_stops",
    "lap_times",
    "sprint_results"
]

# Common helper function for raw ingestion
def read_bronze_csv(file_name):
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("pathGlobFilter", f"{file_name}.csv")
        .load(base_path)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )

# Dynamic Function Generator: Har table ke liye automatically DLT table define karega
def create_bronze_table(table_name):
    @dlt.table(
        name=f"bronze_{table_name}",
        comment=f"Raw ingested {table_name} CSV data from ADLS Gen2",
        table_properties={"quality": "bronze"}
    )
    def bronze_table_func():
        return read_bronze_csv(table_name)
    
    return bronze_table_func

# Loop chalakar saari 14 tables generate karna
for table in f1_tables:
    create_bronze_table(table)